# 01. ANT: the benign floor and the go/no-go gate

**Why this notebook.** The method flags agent traffic that no logged action explains. It can only work if normal
agents send little traffic that their own logged steps do not explain; otherwise no alarm threshold leaves room to
catch anything. This notebook measures that on ANT's benign runs, before any collection harness or attack is built.

**What ANT supports.** Each run has a packet capture and per-message records with a start time and a step type (ANT's
macro group label). The records carry no per-call sizes and no destinations, and they come from the agents' own
session files. Attribution here therefore works at the level of a logged step: the traffic inside each step is
compared with what steps of that type normally send. The per-call byte budget is not tested here.

**Data handling.** Each capture is downloaded, reduced to a packet table (time, flow, direction, payload length) and a
flow table (endpoints, payload bytes each way, destination name from the TLS server name, the HTTP Host header or a
DNS answer in the same capture), and deleted. The tables are in `data/ant/` on Drive and are not committed. Only
destination names are read from payloads; nothing else.

**Steps and parts.** Record i opens a step that lasts until the next record starts; traffic before the first record
is the prelude. Each step falls in one part: *before_record* (the prelude), *coarse* (a step longer than 30 s, where
the record is too coarse to say what happened) or *logged_step* (30 s or shorter).

**Destination classes.** dns (port 53), local (private or loopback address), backend (the agent folder's model
endpoint, found in Stage C and treated as known, as a gateway would know it), common (a registered domain reached in
at least 10% of training episodes), rare (any other named host), unnamed.

**Methods.**
- *Step-level matcher.* From training episodes: for each step type and class, the share of steps of that type that
  upload anything of that class (smoothed), and a byte budget at the 99th percentile. A model call resends the
  conversation so far, so model-endpoint uploads grow through a run: for the model endpoint the budget applies to the
  growth over the largest earlier model-endpoint upload in the same run; for every other class it applies to the
  step's upload. In an evaluated step, a class's upload is unexplained in full if its share is below 5%, otherwise
  only the part above the budget.
- *Per-task allowlist.* A host is allowed for a run if a training run of the same task reached it.
- *Global allowlist.* A registered domain is allowed if any training run reached it.

Both allowlists always allow the model endpoint, DNS and private addresses.

**Evaluation.** The same task instance (`Axxxx`) recurs across agent folders, so a plain leave-one-agent-out split
would show the allowlists the very sites of the evaluated instance. Instances are split into two halves by a fixed
hash; each fold trains on the other five folders' runs in one half and evaluates the held-out folder's runs in the
other half: 6 folders x 2 halves = 12 folds. ANT's own random split is reported alongside.

**Marks, fixed in Stage A before any data is read.**

| mark | what must hold |
|---|---|
| G0: the captures are read | all 3,114 episodes listed with both files; at least 99% extracted without error; in at least 95% of extracted episodes at least 95% of record start times fall inside the capture time span (1 s slack); at least 80% of upload bytes on TCP port 443 carry a destination name. The same checks on a 30-episode probe stop the notebook before the full extraction |
| G1: model endpoints | every agent folder has a named public host reached in at least 95% of its episodes; the one carrying the most upload bytes is the folder's model endpoint |
| M1: the matcher beats the allowlist | pooled over the 12 folds, the matcher leaves a smaller share of benign upload bytes inside logged steps unexplained than the per-task allowlist: the 95% bootstrap interval of the difference (resampling episodes) lies entirely below zero |
| M2: room for a threshold | for the four Claude Code folders pooled, the matcher leaves at most 5% of benign upload bytes inside logged steps unexplained (5% is a judgment threshold fixed here) |
| M3: the step record carries information | pooled over the folds, shuffling step types within each episode raises the matcher's unexplained share inside logged steps: the 95% bootstrap interval of the rise lies entirely above zero |

M3 guards M1 and M2: a matcher that explains everything would pass both and catch nothing.

**Revision after the probe (16 October 2026).** The probe stopped on G0's alignment check. A diagnosis of the probe
captures and of all records showed two things. In every folder except qwen, record times agree with capture times:
the first record comes 1 to 7 s after the capture starts, and the median distance from a record to client upload
traffic is 0.1 to 1.6 s. The qwen folder's records are unusable for timing: 479 of its 625 runs have no records file,
116 carry placeholder start times (the message number read as milliseconds after 1 January 1970), and only 30 runs,
all from one scenario, carry real times. G0 as fixed cannot pass, because its first condition needs every run to have
a records file; it is still evaluated and reported. Written to `reports/ant/revision_01.json` before the full
extraction:
- the decision uses G0r, the same checks with alignment measured on runs whose records carry real times;
- runs without usable records (no records, placeholder times, or record times outside the capture) are left out of
  Stages D and E;
- a folder enters the step-level analysis only if at least half of its runs are usable. This leaves out qwen, so the
  folds are 5 folders x 2 halves = 10. The qwen captures stay in Stage C.

M1, M2 and M3 are unchanged apart from the number of folds.

**Reported, not marked.** Each folder's hosts and model endpoint; the share of traffic in each part; every share per
folder; ANT's own split; simulated injections at a 5% false-interruption threshold.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
!pip install -q dpkt tldextract
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/UNEXPLAINED_TRAFFIC_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'unexplained-traffic')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports/ant', 'figures/ant', 'data/ant/extract'): os.makedirs(d_, exist_ok=True)
print('Session ready. CPUs:', os.cpu_count())

## Stage A. The marks and settings (fixed before any data is read) and the modules

In [ ]:
import json
MARKS = {
    'G0': 'all expected episodes listed with both files; at least 99% extracted without error; in at least 95% of '
          'extracted episodes at least 95% of record start times inside the capture time span (1 s slack); at least '
          '80% of upload bytes on TCP port 443 named',
    'G1': 'every agent folder has a named public host reached in at least 95% of its episodes; the one with the most '
          'upload bytes is its model endpoint',
    'M1': 'pooled over the 12 instance-disjoint leave-one-agent-out folds, 95% bootstrap interval of (matcher minus '
          'per-task allowlist) unexplained share of upload bytes inside logged steps entirely below zero',
    'M2': 'Claude Code folders pooled: matcher unexplained share of upload bytes inside logged steps at most 0.05',
    'M3': 'pooled over the folds, 95% bootstrap interval of (shuffled step types minus true step types) matcher '
          'unexplained share inside logged steps entirely above zero',
}
SETTINGS = dict(expected_episodes=3114, coarse_step_s=30.0, rho=0.05, smoothing=20.0, budget_quantile=0.99,
                min_cells=30, common_share=0.10, alpha=0.05, n_boot=2000, seed=1, align_slack_s=1.0,
                min_extracted=0.99, min_aligned_episodes=0.95, min_aligned_records=0.95, min_named_tcp443=0.80,
                min_endpoint_share=0.95, m2_max=0.05, inject_bytes=[1e4, 1e5, 1e6, 1e7], probe_per_folder=5,
                cc_prefix='cc_')
FIXED = dict(marks=MARKS, settings=SETTINGS)
SETTINGS_PATH = 'reports/ant/settings_01.json'
if os.path.exists(SETTINGS_PATH):
    with open(SETTINGS_PATH) as f:
        assert json.load(f) == FIXED, 'the marks or settings differ from those fixed on the first run'
else:
    with open(SETTINGS_PATH, 'w') as f:
        json.dump(FIXED, f, indent=2)
S = SETTINGS
print(json.dumps(FIXED, indent=2))

### Revision after the probe (16 October 2026)

In [ ]:
REVISION = dict(
    date='2026-10-16',
    reason='the probe stopped on G0 alignment. qwen records are unusable for timing: 479 of 625 runs have no records '
           'file, 116 carry placeholder start times equal to the message number in milliseconds after 1970-01-01, '
           '30 carry real times (all business_services). Record and capture times agree in the other five folders',
    G0r='at least 99% of listed runs extracted without error; among runs whose records carry real times, at least '
        '95% have at least 95% of record start times inside the capture time span (1 s slack); at least 80% of upload '
        'bytes on TCP port 443 named',
    usable_run='extracted; has records; record times after real_time_after; at least 95% of record start times inside '
               'the capture time span (1 s slack)',
    real_time_after='2020-01-01',
    min_usable_folder_share=0.5,
    decision='G0r replaces G0 in the decision and G0 is reported; Stages D and E use usable runs of folders with at '
             'least half their runs usable; M1, M2 and M3 unchanged apart from the number of folds',
)
REV_PATH = 'reports/ant/revision_01.json'
if os.path.exists(REV_PATH):
    with open(REV_PATH) as f:
        assert json.load(f) == REVISION, 'the revision differs from the one written on the first run'
else:
    with open(REV_PATH, 'w') as f:
        json.dump(REVISION, f, indent=2)
print(json.dumps(REVISION, indent=2))

In [ ]:
%%writefile src/ant_capture.py
"""Capture reading for ANT (notebook 01).

Reads one packet capture (pcap or pcapng) and returns two tables and a summary:

  packets  one row per packet that carries transport payload: ts, flow, up (1 = client to server), plen
  flows    one row per bidirectional transport flow: endpoints, protocol, first and last time, payload bytes and
           packets each way, and the destination name (TLS server name, else HTTP Host, else the name that a DNS
           answer in the same capture gave for the server address)

Only metadata is kept. Payload bytes are read only to recover destination names: the first 16 KB a client sends on a
TCP flow (TLS ClientHello server name and ALPN, or an HTTP Host header) and DNS answers. Payload lengths come from the
IP and transport length fields, so Ethernet padding and short snap lengths do not change byte counts.
"""
from __future__ import annotations

import ipaddress
import socket
import struct
from array import array

import dpkt
import numpy as np
import pandas as pd

LINKTYPES = {0: "null", 1: "ethernet", 12: "raw", 14: "raw", 101: "raw", 108: "loop", 113: "sll", 228: "ipv4",
             229: "ipv6", 276: "sll2"}
SERVER_PORTS = {20, 21, 22, 25, 53, 80, 110, 123, 143, 443, 465, 587, 853, 993, 995, 3128, 3306, 5432, 6379, 8080,
                8443, 9418, 27017}
CLIENT_SNIFF_BYTES = 16384
CLIENT_SNIFF_SEGMENTS = 48
TH_SYN, TH_ACK = 0x02, 0x10

_PRIVATE_CACHE: dict[bytes, bool] = {}


def ip_text(raw: bytes) -> str:
    return socket.inet_ntop(socket.AF_INET if len(raw) == 4 else socket.AF_INET6, raw)


def is_private(raw: bytes) -> bool:
    v = _PRIVATE_CACHE.get(raw)
    if v is None:
        a = ipaddress.ip_address(raw)
        v = bool(a.is_private or a.is_loopback or a.is_link_local or a.is_multicast or a.is_unspecified)
        _PRIVATE_CACHE[raw] = v
    return v


def _ip_from_bytes(b: bytes):
    if not b:
        return None
    ver = b[0] >> 4
    if ver == 4:
        return dpkt.ip.IP(b)
    if ver == 6:
        return dpkt.ip6.IP6(b)
    return None


def network_layer(buf: bytes, linktype: int):
    """The IPv4 or IPv6 object inside one frame, or None."""
    kind = LINKTYPES.get(linktype)
    if kind == "ethernet":
        eth = dpkt.ethernet.Ethernet(buf)
        d = eth.data
        return d if isinstance(d, (dpkt.ip.IP, dpkt.ip6.IP6)) else None
    if kind == "sll":
        return _ip_from_bytes(buf[16:]) if len(buf) > 16 else None
    if kind == "sll2":
        return _ip_from_bytes(buf[20:]) if len(buf) > 20 else None
    if kind in ("null", "loop"):
        return _ip_from_bytes(buf[4:]) if len(buf) > 4 else None
    if kind in ("raw", "ipv4", "ipv6"):
        return _ip_from_bytes(buf)
    raise ValueError(f"unsupported link type {linktype}")


def _u16(b: bytes, i: int) -> int:
    return (b[i] << 8) | b[i + 1]


def parse_client_hello(stream: bytes):
    """(server_name, alpn) from the start of a client TLS stream, or (None, None)."""
    hs = b""
    i = 0
    while i + 5 <= len(stream):
        if stream[i] != 22:
            break
        rlen = _u16(stream, i + 3)
        hs += stream[i + 5:i + 5 + rlen]
        i += 5 + rlen
        if len(hs) >= 4 and len(hs) >= 4 + int.from_bytes(hs[1:4], "big"):
            break
    if len(hs) < 4 or hs[0] != 1:
        return None, None
    body = hs[4:4 + int.from_bytes(hs[1:4], "big")]
    sni = alpn = None
    try:
        p = 34
        p += 1 + body[p]
        p += 2 + _u16(body, p)
        p += 1 + body[p]
        if p + 2 > len(body):
            return None, None
        end = min(len(body), p + 2 + _u16(body, p))
        p += 2
        while p + 4 <= end:
            et, el = _u16(body, p), _u16(body, p + 2)
            ed = body[p + 4:p + 4 + el]
            p += 4 + el
            if et == 0 and len(ed) >= 2:
                q = 2
                while q + 3 <= len(ed):
                    nt, nl = ed[q], _u16(ed, q + 1)
                    name = ed[q + 3:q + 3 + nl]
                    q += 3 + nl
                    if nt == 0 and name:
                        sni = name.decode("ascii", "ignore").strip().lower().rstrip(".")
                        break
            elif et == 16 and len(ed) >= 2:
                q, protos = 2, []
                while q < len(ed):
                    ln = ed[q]
                    protos.append(ed[q + 1:q + 1 + ln].decode("ascii", "ignore"))
                    q += 1 + ln
                alpn = ",".join(x for x in protos if x) or None
    except IndexError:
        pass
    return sni, alpn


_HTTP_METHODS = (b"GET ", b"POST ", b"PUT ", b"HEAD ", b"DELETE ", b"PATCH ", b"OPTIONS ", b"CONNECT ")


def parse_http_host(stream: bytes):
    if not stream.startswith(_HTTP_METHODS):
        return None
    head = stream.split(b"\r\n\r\n", 1)[0]
    for line in head.split(b"\r\n")[1:]:
        if line[:5].lower() == b"host:":
            host = line[5:].strip().decode("ascii", "ignore").lower()
            if host.startswith("["):
                return host[1:host.find("]")] if "]" in host else host
            return host.rsplit(":", 1)[0] if host.count(":") == 1 else host
    return None


class _Flow:
    __slots__ = ("idx", "proto", "cip", "cport", "sip", "sport", "first", "last", "up_b", "dn_b", "up_p", "dn_p",
                 "syn", "base", "segs", "seg_bytes", "collect", "dns_start")

    def __init__(self, idx, proto, cip, cport, sip, sport, ts, syn, isn, dns_name):
        self.idx, self.proto = idx, proto
        self.cip, self.cport, self.sip, self.sport = cip, cport, sip, sport
        self.first = self.last = ts
        self.up_b = self.dn_b = self.up_p = self.dn_p = 0
        self.syn = syn
        self.base = ((isn + 1) & 0xFFFFFFFF) if syn else None
        self.segs = {}
        self.seg_bytes = 0
        self.collect = proto == 6
        self.dns_start = dns_name

    def client_stream(self) -> bytes:
        out = bytearray()
        for off in sorted(self.segs):
            if off > len(out):
                break
            data = self.segs[off]
            if off + len(data) > len(out):
                out += data[len(out) - off:]
        return bytes(out[:CLIENT_SNIFF_BYTES])


def _orient(proto, src, sport, dst, dport, flags):
    """True if src is the client."""
    if proto == 6 and flags & TH_SYN:
        return not (flags & TH_ACK)
    if dport in SERVER_PORTS and sport not in SERVER_PORTS:
        return True
    if sport in SERVER_PORTS and dport not in SERVER_PORTS:
        return False
    sp, dp = is_private(src), is_private(dst)
    if sp != dp:
        return sp
    return sport >= dport


def _open(path):
    f = open(path, "rb")
    magic = f.read(4)
    f.seek(0)
    if magic == b"\x0a\x0d\x0d\x0a":
        return f, dpkt.pcapng.Reader(f)
    return f, dpkt.pcap.Reader(f)


def read_capture(path: str):
    """(packets DataFrame, flows DataFrame, info dict) for one capture file."""
    fh, rdr = _open(path)
    linktype = rdr.datalink()
    if linktype not in LINKTYPES:
        fh.close()
        raise ValueError(f"unsupported link type {linktype}")
    flows: dict[tuple, _Flow] = {}
    order: list[_Flow] = []
    dnsmap: dict[bytes, str] = {}
    p_ts, p_flow, p_up, p_len = array("d"), array("i"), array("b"), array("q")
    n = dict(frames=0, ip=0, tcp=0, udp=0, other=0, bad=0, dns_answers=0)
    t_first = t_last = None
    try:
        for ts, buf in rdr:
            n["frames"] += 1
            ts = float(ts)
            if t_first is None:
                t_first = ts
            t_last = ts
            try:
                ip = network_layer(buf, linktype)
            except (dpkt.UnpackError, IndexError, ValueError, struct.error):
                n["bad"] += 1
                continue
            if ip is None:
                n["other"] += 1
                continue
            n["ip"] += 1
            if isinstance(ip, dpkt.ip.IP):
                if ip.offset:
                    n["other"] += 1
                    continue
                proto, l4, ip_payload = ip.p, ip.data, ip.len - ip.hl * 4
            else:
                proto, l4 = ip.p, ip.data
                try:
                    ext = sum(len(h) for h in getattr(ip, "all_extension_headers", []) or [])
                except TypeError:
                    ext = 0
                ip_payload = ip.plen - ext
            src, dst = ip.src, ip.dst
            if proto == 6 and isinstance(l4, dpkt.tcp.TCP):
                n["tcp"] += 1
                sport, dport, flags = l4.sport, l4.dport, l4.flags
                plen = max(0, ip_payload - l4.off * 4)
                data = l4.data if isinstance(l4.data, (bytes, bytearray)) else bytes(l4.data)
                seq = l4.seq
            elif proto == 17 and isinstance(l4, dpkt.udp.UDP):
                n["udp"] += 1
                sport, dport, flags, seq = l4.sport, l4.dport, 0, 0
                plen = max(0, (l4.ulen - 8) if l4.ulen else len(l4.data))
                data = l4.data if isinstance(l4.data, (bytes, bytearray)) else bytes(l4.data)
                if sport == 53 and data:
                    try:
                        d = dpkt.dns.DNS(data)
                        if d.qr == dpkt.dns.DNS_R and d.qd:
                            qname = d.qd[0].name.lower().rstrip(".")
                            for rr in d.an:
                                if rr.type == dpkt.dns.DNS_A and len(getattr(rr, "ip", b"")) == 4:
                                    dnsmap[rr.ip] = qname
                                    n["dns_answers"] += 1
                                elif rr.type == dpkt.dns.DNS_AAAA and len(getattr(rr, "ip6", b"")) == 16:
                                    dnsmap[rr.ip6] = qname
                                    n["dns_answers"] += 1
                    except (dpkt.UnpackError, IndexError, ValueError, struct.error, UnicodeDecodeError):
                        pass
            else:
                n["other"] += 1
                continue
            a, b = (src, sport), (dst, dport)
            key = (proto,) + (a + b if a <= b else b + a)
            fl = flows.get(key)
            if fl is None:
                if _orient(proto, src, sport, dst, dport, flags):
                    cip, cport, sip, sport_s = src, sport, dst, dport
                else:
                    cip, cport, sip, sport_s = dst, dport, src, sport
                syn = proto == 6 and bool(flags & TH_SYN) and not (flags & TH_ACK) and cip == src
                fl = _Flow(len(order), proto, cip, cport, sip, sport_s, ts, syn, seq, dnsmap.get(sip))
                flows[key] = fl
                order.append(fl)
            elif proto == 6 and fl.base is None and (flags & TH_SYN) and not (flags & TH_ACK) and src == fl.cip:
                fl.syn, fl.base = True, (seq + 1) & 0xFFFFFFFF
            fl.last = ts
            up = src == fl.cip and sport == fl.cport
            if plen > 0:
                if up:
                    fl.up_b += plen
                    fl.up_p += 1
                else:
                    fl.dn_b += plen
                    fl.dn_p += 1
                p_ts.append(ts)
                p_flow.append(fl.idx)
                p_up.append(1 if up else 0)
                p_len.append(plen)
                if up and fl.collect and data:
                    if fl.base is None:
                        fl.base = seq
                    off = (seq - fl.base) & 0xFFFFFFFF
                    if off < CLIENT_SNIFF_BYTES:
                        fl.segs[off] = bytes(data)
                        fl.seg_bytes += len(data)
                    if fl.seg_bytes >= CLIENT_SNIFF_BYTES or len(fl.segs) >= CLIENT_SNIFF_SEGMENTS:
                        fl.collect = False
    finally:
        fh.close()

    rows = []
    for fl in order:
        sni = alpn = host = None
        if fl.proto == 6 and fl.segs:
            stream = fl.client_stream()
            if stream[:1] == b"\x16":
                sni, alpn = parse_client_hello(stream)
            else:
                host = parse_http_host(stream)
        dns_name = fl.dns_start or dnsmap.get(fl.sip)
        name, src_ = (sni, "sni") if sni else (host, "http") if host else (dns_name, "dns") if dns_name else ("", "")
        rows.append(dict(flow=fl.idx, proto=fl.proto, client_ip=ip_text(fl.cip), client_port=fl.cport,
                         server_ip=ip_text(fl.sip), server_port=fl.sport, first_ts=fl.first, last_ts=fl.last,
                         up_bytes=fl.up_b, down_bytes=fl.dn_b, up_pkts=fl.up_p, down_pkts=fl.dn_p, syn_seen=fl.syn,
                         sni=sni or "", alpn=alpn or "", http_host=host or "", dns_name=dns_name or "", name=name,
                         name_src=src_, server_private=is_private(fl.sip)))
    flow_cols = ["flow", "proto", "client_ip", "client_port", "server_ip", "server_port", "first_ts", "last_ts",
                 "up_bytes", "down_bytes", "up_pkts", "down_pkts", "syn_seen", "sni", "alpn", "http_host", "dns_name",
                 "name", "name_src", "server_private"]
    flows_df = pd.DataFrame(rows, columns=flow_cols)
    packets = pd.DataFrame({"ts": np.frombuffer(p_ts, dtype=np.float64).copy(),
                            "flow": np.frombuffer(p_flow, dtype=np.intc).astype(np.int32),
                            "up": np.frombuffer(p_up, dtype=np.int8).copy(),
                            "plen": np.frombuffer(p_len, dtype=np.int64).astype(np.int32)})
    tcp443 = (flows_df.proto == 6) & (flows_df.server_port == 443)
    named = flows_df.name != ""
    info = dict(linktype=int(linktype), linktype_name=LINKTYPES[linktype], frames=n["frames"], ip_frames=n["ip"],
                tcp_frames=n["tcp"], udp_frames=n["udp"], other_frames=n["other"], bad_frames=n["bad"],
                dns_answers=n["dns_answers"], first_ts=t_first, last_ts=t_last, flows=len(flows_df),
                payload_packets=len(packets), up_bytes=int(flows_df.up_bytes.sum()),
                down_bytes=int(flows_df.down_bytes.sum()), named_up_bytes=int(flows_df.up_bytes[named].sum()),
                tcp443_up_bytes=int(flows_df.up_bytes[tcp443].sum()),
                tcp443_named_up_bytes=int(flows_df.up_bytes[tcp443 & named].sum()),
                udp443_up_bytes=int(flows_df.up_bytes[(flows_df.proto == 17) & (flows_df.server_port == 443)].sum()))
    return packets, flows_df, info

In [ ]:
%%writefile src/ant_data.py
"""ANT data access (notebook 01): the episode listing, the per-message records, and capture extraction.

The dataset (huggingface.co/datasets/benzenenene/ANT) is laid out as
    <split>/<agent folder>/<scenario>/<Axxxx>/{messages.jsonl, network.pcap}
where Axxxx is the task instance: the first two digits are the task number and the same instance identifier recurs
across agent folders. An episode is one (agent folder, instance) pair.

Extraction runs one episode at a time: download the capture to local disk, read it with ant_capture, write the packet
and flow tables to the project data folder, delete the capture. A small JSON summary is written last and marks the
episode as done, so an interrupted run resumes where it stopped. A job may name a local copy of the dataset
('local_source', the dataset root on disk); captures are then read from it instead of downloaded.
"""
from __future__ import annotations

import json
import multiprocessing as mp
import os
import re
import shutil
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import pandas as pd

import ant_capture

REPO_ID = "benzenenene/ANT"
PATH_RE = re.compile(r"^(train|val|test)/([^/]+)/([^/]+)/(A\d{4})/(network\.pcap|messages\.jsonl)$")
MESSAGE_FIELDS = ["message_index", "message_id", "role", "start_time", "macro_group_id", "macro_group_label",
                  "task_group", "category", "agent", "split"]


def hf_token():
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            return tok
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or None


def list_episodes(token=None, files=None) -> pd.DataFrame:
    """One row per episode with the repository paths of its two files ('' when a file is missing)."""
    if files is None:
        from huggingface_hub import list_repo_files
        files = list_repo_files(REPO_ID, repo_type="dataset", token=token)
    eps: dict[tuple, dict] = {}
    for f in files:
        m = PATH_RE.match(f)
        if not m:
            continue
        split, agent, scenario, inst, fname = m.groups()
        key = (split, agent, scenario, inst)
        row = eps.setdefault(key, dict(split=split, agent=agent, scenario=scenario, instance=inst,
                                       task=int(inst[1:3]), pcap="", messages=""))
        row["pcap" if fname == "network.pcap" else "messages"] = f
    df = pd.DataFrame(list(eps.values()), columns=["split", "agent", "scenario", "instance", "task", "pcap",
                                                   "messages"])
    df["uid"] = df.agent + "/" + df.instance
    return df.sort_values(["agent", "instance"]).reset_index(drop=True)


def to_epoch(series: pd.Series) -> pd.Series:
    t = pd.to_datetime(series, utc=True, format="ISO8601")
    return (t - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds()


def download_messages(local_dir: str, token=None, max_workers: int = 8) -> str:
    from huggingface_hub import snapshot_download
    return snapshot_download(REPO_ID, repo_type="dataset", allow_patterns=["*/messages.jsonl"], local_dir=local_dir,
                             token=token, max_workers=max_workers)


def read_messages(episodes: pd.DataFrame, local_dir: str) -> pd.DataFrame:
    """All per-message records, keyed by the episode identity taken from the folder path."""
    frames = []
    for ep in episodes.itertuples(index=False):
        if not ep.messages:
            continue
        path = Path(local_dir) / ep.messages
        recs = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not recs:
            continue
        d = pd.DataFrame(recs)
        for c in MESSAGE_FIELDS:
            if c not in d.columns:
                d[c] = None
        d = d[MESSAGE_FIELDS].rename(columns={"agent": "record_agent", "split": "record_split"})
        d.insert(0, "uid", ep.uid)
        d.insert(1, "agent", ep.agent)
        d.insert(2, "instance", ep.instance)
        frames.append(d)
    m = pd.concat(frames, ignore_index=True)
    m["t"] = to_epoch(m.start_time)
    if m.t.isna().any():
        print(f"read_messages: {int(m.t.isna().sum())} records without a start time dropped")
        m = m[m.t.notna()]
    m["macro_group_label"] = m.macro_group_label.fillna("unlabelled").astype(str)
    return m.sort_values(["uid", "t", "message_index"]).reset_index(drop=True)


def alignment(messages: pd.DataFrame, infos: pd.DataFrame, slack: float = 1.0) -> pd.Series:
    """Per episode, the share of record start times that fall inside the capture's time span (plus slack seconds)."""
    m = messages.merge(infos[["uid", "first_ts", "last_ts"]], on="uid", how="inner")
    inside = (m.t >= m.first_ts - slack) & (m.t <= m.last_ts + slack)
    return inside.groupby(m.uid).mean()


def episode_paths(out_root: str, uid: str) -> dict:
    agent, inst = uid.split("/")
    base = Path(out_root) / agent / inst
    return dict(packets=f"{base}.packets.parquet", flows=f"{base}.flows.parquet", info=f"{base}.info.json")


def extract_episode(job: dict) -> dict:
    """Download, read and store one capture. Returns the capture summary, or a dict with an 'error' key."""
    uid, paths = job["uid"], episode_paths(job["out_root"], job["uid"])
    if os.path.exists(paths["info"]):
        with open(paths["info"]) as fh:
            return {**json.load(fh), "cached": True}
    if job.get("local_source"):
        local, tmp = os.path.join(job["local_source"], job["pcap"]), None
    else:
        from huggingface_hub import hf_hub_download
        tmp = os.path.join(job["tmp_root"], uid.replace("/", "__"))
        os.makedirs(tmp, exist_ok=True)
        local = None
        for attempt in range(4):
            try:
                local = hf_hub_download(REPO_ID, filename=job["pcap"], repo_type="dataset", local_dir=tmp,
                                        cache_dir=os.path.join(tmp, ".hf_cache"), token=job.get("token"))
                break
            except Exception as e:
                if attempt == 3:
                    shutil.rmtree(tmp, ignore_errors=True)
                    return dict(uid=uid, error=f"download: {e!r}"[:500])
                time.sleep(5 * 2 ** attempt)
    try:
        size = os.path.getsize(local)
        packets, flows, info = ant_capture.read_capture(local)
    except Exception as e:
        return dict(uid=uid, error=f"parse: {e!r}"[:500])
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(os.path.dirname(paths["info"]), exist_ok=True)
    packets.to_parquet(paths["packets"], index=False)
    flows.to_parquet(paths["flows"], index=False)
    info.update(uid=uid, pcap_bytes=size)
    with open(paths["info"], "w") as fh:
        json.dump(info, fh)
    return info


def run_pool(fn, jobs: list, workers: int, every: int = 100, label: str = "") -> list:
    """Run fn over jobs in fresh worker processes, printing a progress line every `every` results."""
    out, t0 = [], time.time()
    ctx = mp.get_context("spawn")
    with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as ex:
        futs = [ex.submit(fn, j) for j in jobs]
        for k, f in enumerate(as_completed(futs), 1):
            out.append(f.result())
            if k % every == 0 or k == len(jobs):
                errs = sum(1 for r in out if isinstance(r, dict) and r.get("error"))
                print(f"{label}{k}/{len(jobs)} done, {errs} errors, {time.time() - t0:.0f} s", flush=True)
    return out

In [ ]:
%%writefile src/ant_floor.py
"""The benign floor on ANT (notebook 01): logged steps, traffic pieces, the step-level matcher and the allowlists.

Steps. The records of an episode are sorted by start time. Record i opens a step that runs until the next record
starts; the last step runs to the end of the capture; traffic before the first record is the 'prelude'. This is ANT's
own segmentation. Each step has the record's macro group label as its type.

Parts. Every step falls in one part: 'before_record' (the prelude), 'coarse' (a step longer than L seconds, where the
record is too coarse to say what happened) or 'logged_step' (a step of at most L seconds).

Pieces. The upload and download payload bytes of one flow inside one step.

Destination classes. dns (server port 53), local (private, loopback or link-local server address), backend (the
agent folder's model endpoint host), common (a named host whose registered domain was reached in at least a set share
of training episodes), rare (any other named host), unnamed (no name recovered).

Step-level matcher, fitted on training episodes only. For each step type g and class c: the share r(g,c) of steps of
type g that upload any bytes of class c (smoothed towards the class's overall share), and a byte budget B(g,c). For the
model endpoint, a model call resends the conversation so far, so its upload grows through a run: the budget applies to
the growth of the step's model-endpoint upload over the largest model-endpoint upload of any earlier step in the same
run. For every other class the budget applies to the step's upload itself. A budget is the upper quantile of the
training values for (g, c) when there are enough of them, else of class c over all step types, else the largest
training value of class c. In an evaluated step, class-c upload is unexplained in full if r(g,c) is below rho, and
otherwise only the part above the budget.

Allowlists (baselines). Per-task: a host is allowed for an episode if a training episode of the same task reached it.
Global: a registered domain is allowed if any training episode reached it. Both always allow the agent's own model
endpoint, DNS and private addresses.
"""
from __future__ import annotations

import hashlib
import math
from functools import lru_cache

import numpy as np
import pandas as pd

CLASSES = ("backend", "dns", "local", "common", "rare", "unnamed")
PARTS = ("before_record", "coarse", "logged_step")

_EXTRACT = None


def _extractor():
    global _EXTRACT
    if _EXTRACT is None:
        import tldextract
        _EXTRACT = tldextract.TLDExtract(suffix_list_urls=(), cache_dir=None)
    return _EXTRACT


@lru_cache(maxsize=None)
def regdomain(name: str) -> str:
    if not name:
        return ""
    r = _extractor()(name)
    dom = getattr(r, "top_domain_under_public_suffix", None)
    if dom is None:
        dom = ".".join(x for x in (r.domain, r.suffix) if x)
    return dom or name


def instance_half(instance: str, salt: str = "unexplained-traffic-01") -> int:
    return int(hashlib.sha256(f"{salt}:{instance}".encode()).hexdigest(), 16) % 2


# ---------------------------------------------------------------- steps and pieces (one episode)

def episode_steps(records: pd.DataFrame, cap_first: float, cap_last: float) -> pd.DataFrame:
    """Steps of one episode. records: rows with t, macro_group_label, role, message_index (any order)."""
    r = records.sort_values(["t", "message_index"])
    t = r.t.to_numpy(dtype=float)
    lo = min(cap_first, t[0]) if len(t) else cap_first
    hi = max(cap_last, t[-1]) if len(t) else cap_last
    starts = np.concatenate([[lo], t])
    ends = np.concatenate([t, [hi + 1e-6]])
    return pd.DataFrame(dict(step=np.arange(len(starts), dtype=np.int32), start=starts, end=ends, dur=ends - starts,
                             type=["prelude"] + r.macro_group_label.astype(str).tolist(),
                             role=["prelude"] + r.role.astype(str).tolist(),
                             message_index=np.concatenate([[-1], r.message_index.to_numpy()]).astype(np.int64)))


def episode_pieces(packets: pd.DataFrame, flows: pd.DataFrame, steps: pd.DataFrame) -> pd.DataFrame:
    """Payload bytes and packets of each flow inside each step (only pairs with traffic)."""
    cols = ["step", "flow", "up_bytes", "down_bytes", "up_pkts", "down_pkts"]
    if packets.empty:
        return pd.DataFrame(columns=cols)
    si = np.searchsorted(steps.start.to_numpy(), packets.ts.to_numpy(), side="right") - 1
    g = pd.DataFrame(dict(step=np.clip(si, 0, None).astype(np.int32), flow=packets.flow.to_numpy(),
                          up=packets.up.to_numpy(), plen=packets.plen.to_numpy(dtype=np.int64)))
    a = g.groupby(["step", "flow", "up"]).plen.agg(["sum", "count"]).unstack("up", fill_value=0)
    out = pd.DataFrame(index=a.index)
    for d, name in ((1, "up"), (0, "down")):
        out[f"{name}_bytes"] = a[("sum", d)] if ("sum", d) in a.columns else 0
        out[f"{name}_pkts"] = a[("count", d)] if ("count", d) in a.columns else 0
    return out.reset_index()[cols].astype({c: np.int64 for c in cols[2:]})


def pieces_job(job: dict) -> dict:
    """Worker: steps and pieces of one episode from its stored packet and flow tables."""
    packets = pd.read_parquet(job["packets"])
    flows = pd.read_parquet(job["flows"])
    recs = pd.DataFrame(job["records"])
    steps = episode_steps(recs, job["cap_first"], job["cap_last"])
    pieces = episode_pieces(packets, flows, steps)
    keep = ["flow", "proto", "server_ip", "server_port", "name", "server_private"]
    pieces = pieces.merge(flows[keep], on="flow", how="left")
    steps.insert(0, "uid", job["uid"])
    pieces.insert(0, "uid", job["uid"])
    return dict(uid=job["uid"], steps=steps, pieces=pieces)


# ---------------------------------------------------------------- classes, parts, step-class table

def add_hosts(pieces: pd.DataFrame) -> pd.DataFrame:
    p = pieces.copy()
    p["host"] = np.where(p.name.astype(str) != "", p.name.astype(str), p.server_ip.astype(str))
    names = p.name.astype(str)
    uniq = {n: regdomain(n) for n in names.unique()}
    p["regdom"] = names.map(uniq)
    p.loc[p.regdom == "", "regdom"] = p.loc[p.regdom == "", "server_ip"].astype(str)
    return p


def classify(pieces: pd.DataFrame, backend: dict, common: set) -> np.ndarray:
    named = pieces.name.astype(str) != ""
    is_backend = pieces.host.to_numpy() == pieces.agent.map(backend).fillna("").to_numpy()
    return np.select([pieces.server_port.to_numpy() == 53, pieces.server_private.to_numpy(dtype=bool), is_backend,
                      named.to_numpy() & pieces.regdom.isin(common).to_numpy(), named.to_numpy()],
                     ["dns", "local", "backend", "common", "rare"], "unnamed")


def add_parts(steps: pd.DataFrame, L: float) -> pd.DataFrame:
    s = steps.copy()
    s["part"] = np.where(s.type == "prelude", "before_record", np.where(s.dur > L, "coarse", "logged_step"))
    return s


def common_domains(pieces: pd.DataFrame, uids, share: float) -> set:
    p = pieces[pieces.uid.isin(set(uids)) & (pieces.name.astype(str) != "")]
    n = len(set(uids))
    counts = p.groupby("regdom").uid.nunique()
    return set(counts[counts >= share * n].index)


def prior_max(uid: np.ndarray, step: np.ndarray, up: np.ndarray) -> np.ndarray:
    """For rows of one class sorted by (uid, step): the largest upload of any earlier row of the same run (0 if none)."""
    out = np.zeros(len(up), dtype=float)
    if len(up) == 0:
        return out
    starts = np.r_[True, uid[1:] != uid[:-1]]
    run_max = 0.0
    for i in range(len(up)):
        if starts[i]:
            run_max = 0.0
        out[i] = run_max
        run_max = max(run_max, float(up[i]))
    return out


def step_class_table(pieces: pd.DataFrame, steps: pd.DataFrame) -> pd.DataFrame:
    """One row per (uid, step, class) with upload bytes, joined to the step's type, duration and part. 'prior' is the
    largest upload of the same class in any earlier step of the run (used for the model endpoint only)."""
    sc = pieces.groupby(["uid", "step", "cls"], as_index=False).up_bytes.sum().rename(columns={"up_bytes": "up"})
    sc = sc.sort_values(["cls", "uid", "step"]).reset_index(drop=True)
    sc["prior"] = 0.0
    b = (sc.cls == "backend").to_numpy()
    sc.loc[b, "prior"] = prior_max(sc.uid.to_numpy()[b], sc.step.to_numpy()[b], sc.up.to_numpy(dtype=float)[b])
    return sc.merge(steps[["uid", "step", "type", "dur", "part"]], on=["uid", "step"], how="left")


# ---------------------------------------------------------------- the matcher

class StepMatcher:
    GROWTH = "backend"

    def __init__(self, rho=0.05, smoothing=20.0, quantile=0.99, min_cells=30):
        self.rho, self.A, self.q, self.min_cells = rho, smoothing, quantile, min_cells

    def _budget(self, v: pd.Series) -> float:
        return float(v.quantile(self.q)) if len(v) >= self.min_cells else float(v.max())

    def fit(self, steps: pd.DataFrame, sc: pd.DataFrame):
        live = steps[steps.dur > 0]
        n_g = live.groupby("type").size()
        pos = sc[sc.up > 0].copy()
        pos["v"] = np.where(pos.cls == self.GROWTH, pos.up - pos.prior, pos.up).astype(float)
        k_gc = pos.groupby(["type", "cls"]).size()
        self.r_glob = (pos.groupby("cls").size() / max(len(live), 1)).reindex(list(CLASSES), fill_value=0.0)
        rates = {}
        for (g, c), k in k_gc.items():
            rates[(g, c)] = (k + self.A * self.r_glob[c]) / (n_g.get(g, 0) + self.A)
        for g, n in n_g.items():
            for c in CLASSES:
                if (g, c) not in rates:
                    rates[(g, c)] = (self.A * self.r_glob[c]) / (n + self.A)
        self.rates = rates
        self.budget_glob = {c: self._budget(v) for c, v in pos.groupby("cls").v}
        self.budget = {(g, c): float(v.quantile(self.q)) for (g, c), v in pos.groupby(["type", "cls"]).v
                       if len(v) >= self.min_cells}
        return self

    def rate(self, g, c):
        return self.rates.get((g, c), float(self.r_glob.get(c, 0.0)))

    def budget_of(self, g, c):
        return self.budget.get((g, c), self.budget_glob.get(c, 0.0))

    def unexplained(self, types, classes, up, prior=None) -> np.ndarray:
        keys = list(zip(types, classes))
        uniq = {k: (self.rate(*k), self.budget_of(*k)) for k in set(keys)}
        r = np.array([uniq[k][0] for k in keys], dtype=float)
        b = np.maximum(np.array([uniq[k][1] for k in keys], dtype=float), 0.0)
        up = np.asarray(up, dtype=float)
        base = np.zeros(len(up)) if prior is None else np.asarray(prior, dtype=float)
        base = np.where(np.asarray(classes) == self.GROWTH, base, 0.0)
        return np.where(r >= self.rho, np.maximum(0.0, up - base - b), up)


# ---------------------------------------------------------------- the allowlists

def allow_per_task(train_pieces: pd.DataFrame) -> set:
    return set(zip(train_pieces.task.to_numpy(), train_pieces.host.to_numpy()))


def allow_global(train_pieces: pd.DataFrame) -> set:
    return set(train_pieces.regdom.unique())


def allowlist_unexplained(pieces: pd.DataFrame, allowed: set, per_task: bool) -> np.ndarray:
    always = pieces.cls.isin(["backend", "dns", "local"]).to_numpy()
    if per_task:
        ok = np.fromiter(((t, h) in allowed for t, h in zip(pieces.task.to_numpy(), pieces.host.to_numpy())),
                         dtype=bool, count=len(pieces))
    else:
        ok = pieces.regdom.isin(allowed).to_numpy()
    return np.where(always | ok, 0.0, pieces.up_bytes.to_numpy(dtype=float))


# ---------------------------------------------------------------- one evaluation (a fold, or ANT's own split)

def evaluate(pieces: pd.DataFrame, steps: pd.DataFrame, backend: dict, train_uids, eval_uids, S: dict, seed: int):
    """Fit on train_uids, score eval_uids. Returns (per-step table of upload and unexplained bytes per method,
    the fitted matcher, the evaluated step-class rows, the evaluated steps)."""
    tr, ev = set(train_uids), set(eval_uids)
    P = pieces[pieces.uid.isin(tr | ev)].copy()
    common = common_domains(P, tr, S["common_share"])
    P["cls"] = classify(P, backend, common)
    sc = step_class_table(P, steps[steps.uid.isin(tr | ev)])
    m = StepMatcher(S["rho"], S["smoothing"], S["budget_quantile"], S["min_cells"])
    m.fit(steps[steps.uid.isin(tr)], sc[sc.uid.isin(tr)])
    sce = sc[sc.uid.isin(ev)].copy()
    sce["u_matcher"] = m.unexplained(sce.type.to_numpy(), sce.cls.to_numpy(), sce.up.to_numpy(), sce.prior.to_numpy())
    ste = steps[steps.uid.isin(ev)].copy()
    sh = shuffle_types(ste, seed)[["uid", "step", "type"]].rename(columns={"type": "type_shuffled"})
    sce = sce.merge(sh, on=["uid", "step"], how="left")
    sce["u_shuffled"] = m.unexplained(sce.type_shuffled.to_numpy(), sce.cls.to_numpy(), sce.up.to_numpy(),
                                      sce.prior.to_numpy())
    pe, pt = P[P.uid.isin(ev)].copy(), P[P.uid.isin(tr)]
    pe["u_task"] = allowlist_unexplained(pe, allow_per_task(pt), True)
    pe["u_global"] = allowlist_unexplained(pe, allow_global(pt), False)
    a1 = pe.groupby(["uid", "step"], as_index=False)[["up_bytes", "u_task", "u_global"]].sum()
    a1 = a1.rename(columns={"up_bytes": "up"})
    a2 = sce.groupby(["uid", "step"], as_index=False)[["u_matcher", "u_shuffled"]].sum()
    t = a1.merge(a2, on=["uid", "step"], how="outer")
    for c in ("up", "u_task", "u_global", "u_matcher", "u_shuffled"):
        t[c] = t[c].fillna(0.0).astype(float)
    t = t.merge(ste[["uid", "step", "part", "type", "dur"]], on=["uid", "step"], how="left")
    return t, m, sce, ste


# ---------------------------------------------------------------- shares and intervals

def episode_sums(df: pd.DataFrame, cols) -> pd.DataFrame:
    """Per (uid, part): upload bytes and the unexplained bytes of each method column."""
    return df.groupby(["uid", "agent", "part"], as_index=False)[["up"] + list(cols)].sum()


def shares(ep: pd.DataFrame, cols, by=None) -> pd.DataFrame:
    keys = ([by] if by else []) + ["part"]
    g = ep.groupby(keys)[["up"] + list(cols)].sum()
    tot = ep.groupby([by] if by else lambda _: "all")[["up"] + list(cols)].sum()
    out = []
    for k, row in g.iterrows():
        k = k if isinstance(k, tuple) else (k,)
        out.append(dict(zip(keys, k), up_bytes=row.up, **{f"{c}_share": (row[c] / row.up if row.up else np.nan)
                                                          for c in cols}))
    for k, row in tot.iterrows():
        d = {by: k} if by else {}
        out.append(dict(d, part="all", up_bytes=row.up, **{f"{c}_share": (row[c] / row.up if row.up else np.nan)
                                                            for c in cols}))
    return pd.DataFrame(out)


def boot_ratio_diff(ep: pd.DataFrame, a: str, b: str, n_boot=2000, seed=1):
    """Point estimate and 95% interval of sum(a)/sum(up) - sum(b)/sum(up), resampling episodes."""
    e = ep.groupby("uid")[["up", a, b]].sum()
    up, A, B = e.up.to_numpy(float), e[a].to_numpy(float), e[b].to_numpy(float)
    point = (A.sum() - B.sum()) / up.sum()
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(e), size=(n_boot, len(e)))
    reps = (A[idx].sum(1) - B[idx].sum(1)) / up[idx].sum(1)
    lo, hi = np.percentile(reps, [2.5, 97.5])
    return float(point), float(lo), float(hi)


def boot_ratio(ep: pd.DataFrame, a: str, n_boot=2000, seed=1):
    e = ep.groupby("uid")[["up", a]].sum()
    up, A = e.up.to_numpy(float), e[a].to_numpy(float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(e), size=(n_boot, len(e)))
    reps = A[idx].sum(1) / up[idx].sum(1)
    lo, hi = np.percentile(reps, [2.5, 97.5])
    return float(A.sum() / up.sum()), float(lo), float(hi)


def conformal_threshold(scores, alpha: float) -> float:
    s = np.sort(np.asarray(scores, dtype=float))
    k = math.ceil((len(s) + 1) * (1 - alpha))
    return float(s[k - 1]) if 1 <= k <= len(s) else float("inf")


def shuffle_types(steps: pd.DataFrame, seed: int) -> pd.DataFrame:
    """Permute step types among the record steps (not the prelude) of each episode."""
    s = steps.copy()
    rng = np.random.default_rng(seed)
    idx = np.flatnonzero((s.type != "prelude").to_numpy())
    types = s.type.to_numpy().copy()
    uids = s.uid.to_numpy()
    for u in pd.unique(uids[idx]):
        pos = idx[uids[idx] == u]
        types[pos] = types[rng.permutation(pos)]
    s["type"] = types
    return s


# ---------------------------------------------------------------- simulated injections

def pick_steps(steps_ev: pd.DataFrame, sc_ev: pd.DataFrame, which: str, seed: int) -> pd.DataFrame:
    """One logged step per run, chosen at random: 'backend' = a logged step with model-endpoint upload,
    'any' = any logged step of positive duration."""
    s = steps_ev[(steps_ev.part == "logged_step") & (steps_ev.dur > 0)][["uid", "step", "type"]]
    if which == "backend":
        has = sc_ev[(sc_ev.cls == "backend") & (sc_ev.up > 0) & (sc_ev.part == "logged_step")]
        s = s.merge(has[["uid", "step"]].drop_duplicates(), on=["uid", "step"])
    rng = np.random.default_rng(seed)
    return s.iloc[rng.permutation(len(s))].drop_duplicates("uid").reset_index(drop=True)


def injection_delta(matcher: StepMatcher, sc_ev: pd.DataFrame, picks: pd.DataFrame, cls: str, X: float):
    """Rise in each picked run's unexplained bytes when X upload bytes of class cls are added to its picked step.
    For the model endpoint the whole run is rescored, since a larger upload also raises the reference for later
    steps."""
    if cls != matcher.GROWTH:
        cur = sc_ev[sc_ev.cls == cls].groupby(["uid", "step"]).up.sum()
        u0 = cur.reindex(pd.MultiIndex.from_frame(picks[["uid", "step"]])).fillna(0).to_numpy(dtype=float)
        t, c = picks.type.to_numpy(), np.full(len(picks), cls)
        return matcher.unexplained(t, c, u0 + X) - matcher.unexplained(t, c, u0)
    runs = {u: d.sort_values("step") for u, d in sc_ev[sc_ev.cls == cls].groupby("uid")}
    out = np.zeros(len(picks))
    for k, (u, st) in enumerate(picks[["uid", "step"]].itertuples(index=False)):
        d = runs[u]
        steps, types, up = d.step.to_numpy(), d.type.to_numpy(), d.up.to_numpy(dtype=float)
        up2 = up.copy()
        up2[np.flatnonzero(steps == st)[0]] += X
        c = np.full(len(up), cls)
        uid = np.full(len(up), u)
        before = matcher.unexplained(types, c, up, prior_max(uid, steps, up)).sum()
        after = matcher.unexplained(types, c, up2, prior_max(uid, steps, up2)).sum()
        out[k] = after - before
    return out

In [ ]:
import importlib, glob
import numpy as np, pandas as pd
import huggingface_hub
from packaging.version import Version
import ant_capture, ant_data, ant_floor, reslog
for m_ in (ant_capture, ant_data, ant_floor, reslog):
    importlib.reload(m_)
assert Version(huggingface_hub.__version__) >= Version('0.23'), huggingface_hub.__version__
DATA = 'data/ant'
EXTRACT = os.path.join(REPO_DIR, DATA, 'extract')
TMP = '/content/ant_tmp'
WORKERS = min(8, max(4, 2 * (os.cpu_count() or 2)))
TOKEN = ant_data.hf_token()
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
print('huggingface_hub', huggingface_hub.__version__, '| workers', WORKERS, '| HF token', 'yes' if TOKEN else 'no')

## Stage B. The episode listing, the records and the captures (resumable)

In [ ]:
EP = ant_data.list_episodes(TOKEN)
assert not EP.duplicated('uid').any(), 'an (agent folder, instance) pair occurs more than once'
EP.to_parquet(f'{DATA}/episodes_01.parquet', index=False)
BOTH = (EP.pcap != '') & (EP.messages != '')
print(f'{len(EP)} episodes listed, {int(BOTH.sum())} with both files (expected {S["expected_episodes"]})')
display(EP.pivot_table(index='agent', columns='split', values='uid', aggfunc='count', fill_value=0, margins=True))
display(EP.groupby('scenario').task.agg(lambda x: sorted(set(x))).to_frame('tasks'))

In [ ]:
MSG_PATH = f'{DATA}/messages_01.parquet'
if os.path.exists(MSG_PATH):
    MSG = pd.read_parquet(MSG_PATH)
else:
    ant_data.download_messages('/content/ant_messages', TOKEN)
    MSG = ant_data.read_messages(EP, '/content/ant_messages')
    MSG.to_parquet(MSG_PATH, index=False)
MSG['gap_s'] = MSG.groupby('uid').t.diff()
rec = MSG.groupby('agent').agg(episodes=('uid', 'nunique'), records=('t', 'size'), step_types=('macro_group_label', 'nunique'),
                               median_gap_s=('gap_s', 'median'), gaps_over_30s=('gap_s', lambda g: float((g > S['coarse_step_s']).mean())))
rec['records_per_episode'] = rec.records / rec.episodes
print(f'{len(MSG)} records in {MSG.uid.nunique()} episodes; {MSG.macro_group_label.nunique()} step types')
display(rec.round(3))

In [ ]:
REAL_T = pd.Timestamp(REVISION['real_time_after'], tz='UTC').timestamp()
FIRST_T = MSG.groupby('uid').t.min()
PLACEHOLDER = set(FIRST_T[FIRST_T < REAL_T].index)
HAS_REC = set(MSG.uid)
MSG_REAL = MSG[~MSG.uid.isin(PLACEHOLDER)]
print(f'runs with records: {len(HAS_REC)}; with placeholder record times: {len(PLACEHOLDER)}')
display(EP.assign(records=EP.uid.isin(HAS_REC), placeholder=EP.uid.isin(PLACEHOLDER))
          .groupby('agent')[['records', 'placeholder']].sum().assign(runs=EP.groupby('agent').size()))

def capture_checks(R, n_listed):
    err = R['error'].notna() if 'error' in R.columns else pd.Series(False, index=R.index)
    ok = R[~err]
    al = ant_data.alignment(MSG, ok, S['align_slack_s'])
    al_real = ant_data.alignment(MSG_REAL, ok, S['align_slack_s'])
    tcp443 = ok.tcp443_up_bytes.sum()
    share = lambda a: float((a >= S['min_aligned_records']).mean()) if len(a) else float('nan')
    return dict(listed=n_listed, extracted=len(ok), errors=int(err.sum()), extracted_share=len(ok) / max(n_listed, 1),
                no_records=int((~ok.uid.isin(HAS_REC)).sum()), placeholder_times=int(ok.uid.isin(PLACEHOLDER).sum()),
                aligned_episode_share=share(al), aligned_episode_share_real_times=share(al_real),
                named_tcp443_share=float(ok.tcp443_named_up_bytes.sum() / tcp443) if tcp443 else float('nan'),
                linktypes=ok.linktype_name.value_counts().to_dict() if len(ok) else {})

JOBS = {r.uid: dict(uid=r.uid, pcap=r.pcap, out_root=EXTRACT, tmp_root=TMP, token=TOKEN)
        for r in EP[EP.pcap != ''].itertuples()}
probe_uids = [u for _, d in EP[EP.pcap != ''].groupby('agent')
              for u in d.sample(min(len(d), S['probe_per_folder']), random_state=S['seed']).uid]
PR = pd.DataFrame(ant_data.run_pool(ant_data.extract_episode, [JOBS[u] for u in probe_uids], WORKERS, every=10,
                                    label='probe: '))
PROBE = capture_checks(PR, len(probe_uids))
print(json.dumps(PROBE, indent=2))
if 'error' in PR.columns and PR.error.notna().any():
    display(PR[PR.error.notna()][['uid', 'error']])
assert PROBE['extracted_share'] >= S['min_extracted'], 'probe: captures fail to extract'
assert PROBE['aligned_episode_share_real_times'] >= S['min_aligned_episodes'],     'probe: record times and capture times do not align (runs with real record times)'
assert PROBE['named_tcp443_share'] >= S['min_named_tcp443'], 'probe: too few destination names recovered'

In [ ]:
RES = pd.DataFrame(ant_data.run_pool(ant_data.extract_episode, [JOBS[u] for u in EP.uid if u in JOBS], WORKERS,
                                     every=100, label='extract: '))
ERR = RES[RES.error.notna()] if 'error' in RES.columns else RES.iloc[0:0]
INFO = RES[~RES.uid.isin(ERR.uid)].drop(columns=[c for c in ('error', 'cached') if c in RES.columns])
G0V = capture_checks(RES, len(EP))
G0V['listed_with_both'] = int(BOTH.sum())
G0_PASS = bool(len(EP) == S['expected_episodes'] and BOTH.all() and G0V['extracted_share'] >= S['min_extracted']
               and G0V['aligned_episode_share'] >= S['min_aligned_episodes']
               and G0V['named_tcp443_share'] >= S['min_named_tcp443'])
G0R_PASS = bool(G0V['extracted_share'] >= S['min_extracted']
                and G0V['aligned_episode_share_real_times'] >= S['min_aligned_episodes']
                and G0V['named_tcp443_share'] >= S['min_named_tcp443'])
ALIGN = ant_data.alignment(MSG, INFO, S['align_slack_s']).rename('aligned_records')
US = EP[['uid', 'agent']].copy()
US['extracted'] = US.uid.isin(set(INFO.uid))
US['records'] = US.uid.isin(HAS_REC)
US['placeholder_times'] = US.uid.isin(PLACEHOLDER)
US['aligned'] = US.uid.map(ALIGN).fillna(0.0) >= S['min_aligned_records']
US['usable'] = US.extracted & US.records & ~US.placeholder_times & US.aligned
USAGE = US.groupby('agent')[['extracted', 'records', 'placeholder_times', 'aligned', 'usable']].sum()
USAGE.insert(0, 'runs', US.groupby('agent').size())
USAGE['usable_share'] = USAGE.usable / USAGE.runs
USAGE['in_step_analysis'] = USAGE.usable_share >= REVISION['min_usable_folder_share']
USAGE.to_csv('reports/ant/usable_runs_01.csv')
REP = (EP.merge(INFO, on='uid', how='left').merge(ALIGN, left_on='uid', right_index=True, how='left')
         .merge(MSG.groupby('uid').size().rename('records'), left_on='uid', right_index=True, how='left'))
REP.drop(columns=['pcap', 'messages']).to_csv('reports/ant/episodes_01.csv', index=False)
print(json.dumps(G0V, indent=2))
print('G0 (as fixed):', 'PASS' if G0_PASS else 'FAIL', '| G0r (revision 01):', 'PASS' if G0R_PASS else 'FAIL')
display(USAGE.round(3))
if len(ERR):
    display(ERR[['uid', 'error']])
display(REP.groupby('agent')[['pcap_bytes', 'frames', 'flows', 'up_bytes', 'down_bytes', 'aligned_records']].median())

## Stage C. The model endpoint of each agent folder

In [ ]:
from concurrent.futures import ThreadPoolExecutor
FLOWS_PATH = f'{DATA}/flows_01.parquet'
if os.path.exists(FLOWS_PATH):
    FL = pd.read_parquet(FLOWS_PATH)
else:
    def load_flows(u):
        f = pd.read_parquet(ant_data.episode_paths(EXTRACT, u)['flows'])
        f.insert(0, 'uid', u)
        return f
    with ThreadPoolExecutor(16) as ex:
        FL = pd.concat(list(ex.map(load_flows, INFO.uid)), ignore_index=True)
    FL.to_parquet(FLOWS_PATH, index=False)
FL = FL[FL.uid.isin(set(INFO.uid))].merge(EP[['uid', 'agent']], on='uid')
N_EP = EP[EP.uid.isin(set(INFO.uid))].groupby('agent').uid.nunique()
pub = FL[(FL.name != '') & ~FL.server_private & (FL.server_port != 53)]
H = pub.groupby(['agent', 'name']).agg(episodes=('uid', 'nunique'), flows=('flow', 'size'), up_bytes=('up_bytes', 'sum'),
                                     down_bytes=('down_bytes', 'sum')).reset_index()
H['episode_share'] = H.episodes / H.agent.map(N_EP)
H['up_share'] = H.up_bytes / H.groupby('agent').up_bytes.transform('sum')
H = H.sort_values(['agent', 'up_bytes'], ascending=[True, False])
H.groupby('agent').head(10).to_csv('reports/ant/agent_hosts_01.csv', index=False)
CAND = H[H.episode_share >= S['min_endpoint_share']]
BACKEND = CAND.sort_values('up_bytes', ascending=False).drop_duplicates('agent').set_index('agent').name.to_dict()
BK = pd.DataFrame([dict(agent=a, episodes=int(N_EP[a]), model_endpoint=BACKEND.get(a, ''),
                        endpoint_up_share=float(H[(H.agent == a) & (H.name == BACKEND.get(a, ''))].up_share.sum()),
                        always_reached=', '.join(CAND[CAND.agent == a].name.head(6))) for a in N_EP.index])
BK.to_csv('reports/ant/agent_backends_01.csv', index=False)
G1_PASS = bool(all(BACKEND.get(a) for a in N_EP.index))
display(BK); print('G1', 'PASS' if G1_PASS else 'FAIL')
display(H.groupby('agent').head(6).round(3))

## Stage D. Steps, pieces, the matcher and the allowlists

In [ ]:
STEPS_PATH, PIECES_PATH = f'{DATA}/steps_01.parquet', f'{DATA}/pieces_01.parquet'
if os.path.exists(PIECES_PATH):
    ST, PC = pd.read_parquet(STEPS_PATH), pd.read_parquet(PIECES_PATH)
else:
    cols = ['t', 'macro_group_label', 'role', 'message_index']
    recs = {u: d[cols].to_dict('list') for u, d in MSG.groupby('uid')}
    empty = {c: [] for c in cols}
    pjobs = [dict(uid=r.uid, **{k: v for k, v in ant_data.episode_paths(EXTRACT, r.uid).items() if k != 'info'},
                  records=recs.get(r.uid, empty), cap_first=r.first_ts, cap_last=r.last_ts) for r in INFO.itertuples()]
    out = ant_data.run_pool(ant_floor.pieces_job, pjobs, WORKERS, every=200, label='pieces: ')
    ST = pd.concat([o['steps'] for o in out], ignore_index=True)
    PC = pd.concat([o['pieces'] for o in out], ignore_index=True)
    ST.to_parquet(STEPS_PATH, index=False); PC.to_parquet(PIECES_PATH, index=False)
EPA = EP.set_index('uid')
for c in ('agent', 'task', 'instance', 'split'):
    ST[c] = ST.uid.map(EPA[c]); PC[c] = PC.uid.map(EPA[c])
ST = ant_floor.add_parts(ST, S['coarse_step_s'])
PC = ant_floor.add_hosts(PC).merge(ST[['uid', 'step', 'part']], on=['uid', 'step'], how='left')
WHERE = PC.pivot_table(index='agent', columns='part', values='up_bytes', aggfunc='sum', fill_value=0)
WHERE = WHERE.div(WHERE.sum(axis=1), axis=0)
WHERE.to_csv('reports/ant/upload_by_part_01.csv')
print(f'{len(ST)} steps and {len(PC)} pieces in {ST.uid.nunique()} episodes')
print('share of upload bytes in each part:'); display(WHERE.round(3))

In [ ]:
STEP_FOLDERS = set(USAGE.index[USAGE.in_step_analysis])
UIDS = sorted(US[US.usable & US.agent.isin(STEP_FOLDERS)].uid)
print(f'{len(UIDS)} usable runs in {len(STEP_FOLDERS)} folders enter Stages D and E; left out:',
      {a: int(n) for a, n in (USAGE.runs - USAGE.usable.where(USAGE.in_step_analysis, 0)).items()})
HALF = {u: ant_floor.instance_half(EPA.at[u, 'instance']) for u in UIDS}
AGENTS = sorted(EPA.loc[UIDS, 'agent'].unique())
EV, FOLDS = [], []
for a in AGENTS:
    for h in (0, 1):
        tr = [u for u in UIDS if EPA.at[u, 'agent'] != a and HALF[u] != h]
        ev = [u for u in UIDS if EPA.at[u, 'agent'] == a and HALF[u] == h]
        if not ev:
            continue
        t, m, sce, ste = ant_floor.evaluate(PC, ST, BACKEND, tr, ev, S, seed=S['seed'] + len(FOLDS))
        t['fold'], t['agent'] = f'{a}|{h}', a
        EV.append(t)
        FOLDS.append(dict(fold=f'{a}|{h}', agent=a, half=h, n_train=len(tr), n_eval=len(ev), matcher=m, sce=sce, ste=ste))
        print(f'fold {a}|{h}: train {len(tr)} episodes, evaluate {len(ev)}')
EV = pd.concat(EV, ignore_index=True)
print(f'{len(FOLDS)} folds')
tr_ant = [u for u in UIDS if EPA.at[u, 'split'] == 'train']
ev_ant = [u for u in UIDS if EPA.at[u, 'split'] == 'test']
EV_ANT, _, _, _ = ant_floor.evaluate(PC, ST, BACKEND, tr_ant, ev_ant, S, seed=S['seed'])
EV_ANT['agent'] = EV_ANT.uid.map(EPA.agent)
print(f"ANT's own split: train {len(tr_ant)} episodes, evaluate {len(ev_ant)}")

In [ ]:
COLS = ['u_matcher', 'u_task', 'u_global', 'u_shuffled']
EPS = ant_floor.episode_sums(EV, COLS)
LOG = EPS[EPS.part == 'logged_step']
CC = LOG[LOG.agent.str.contains(S['cc_prefix'])]
M1 = ant_floor.boot_ratio_diff(LOG, 'u_matcher', 'u_task', S['n_boot'], S['seed'])
M2 = ant_floor.boot_ratio(CC, 'u_matcher', S['n_boot'], S['seed'])
M3 = ant_floor.boot_ratio_diff(LOG, 'u_shuffled', 'u_matcher', S['n_boot'], S['seed'])
FLOOR = pd.concat([ant_floor.shares(EPS, COLS, by='agent'),
                   ant_floor.shares(EPS, COLS).assign(agent='pooled'),
                   ant_floor.shares(EPS[EPS.agent.str.contains(S['cc_prefix'])], COLS).assign(agent='claude_code')],
                  ignore_index=True)
FLOOR.to_csv('reports/ant/floor_01.csv', index=False)
EPS_ANT = ant_floor.episode_sums(EV_ANT, COLS[:3])
ANTP = pd.concat([ant_floor.shares(EPS_ANT, COLS[:3]).assign(protocol="ANT's own split"),
                  ant_floor.shares(EPS, COLS[:3]).assign(protocol='instance-disjoint leave-one-agent-out')])
ANTP.to_csv('reports/ant/ant_protocol_01.csv', index=False)
print('M1 matcher minus per-task allowlist, logged steps: %.4f [%.4f, %.4f]' % M1)
print('M2 matcher share, Claude Code folders, logged steps: %.4f [%.4f, %.4f]' % M2)
print('M3 shuffled minus true step types, logged steps: %.4f [%.4f, %.4f]' % M3)
show = FLOOR[FLOOR.part.isin(['logged_step', 'all'])].pivot_table(index='agent', columns='part',
                                                                  values=[c + '_share' for c in COLS])
display(show.round(4))
display(ANTP[ANTP.part.isin(['logged_step', 'all'])].round(4))

## Stage E. Simulated injections (reported, not marked)

Within each fold, the held-out runs are split by a second instance hash. On one half (calibration) each method's
threshold is set so that at most 5% of benign runs would be interrupted (the conformal quantile of the per-run
unexplained upload bytes). On the other half (test) each run receives X extra upload bytes, either inside the model
call of one random logged step (*model_call*) or to a never-seen host in one random logged step (*new_host*). A run is
caught when its unexplained bytes pass the threshold. ANT's runs are benign, so this simulates the arithmetic of the
methods on real benign traffic; it is not an attack result.

In [ ]:
rows = []
METHODS = ['u_matcher', 'u_task', 'u_global']
for k, F in enumerate(FOLDS):
    score = EV[EV.fold == F['fold']].groupby('uid')[METHODS].sum()
    cal = np.array([ant_floor.instance_half(EPA.at[u, 'instance'], 'calibration') == 0 for u in score.index])
    if cal.sum() < 2 or (~cal).sum() < 1:
        continue
    lam = {c: ant_floor.conformal_threshold(score.loc[cal, c], S['alpha']) for c in METHODS}
    test = score.loc[~cal]
    common = dict(fold=F['fold'], agent=F['agent'], cal_runs=int(cal.sum()))
    for c in METHODS:
        rows.append(dict(common, method=c, threshold=lam[c], attack='none', bytes=0.0, runs=len(test),
                         caught=int((test[c] > lam[c]).sum())))
    for attack, which, cls in (('model_call', 'backend', 'backend'), ('new_host', 'any', 'rare')):
        picks = ant_floor.pick_steps(F['ste'][F['ste'].uid.isin(test.index)], F['sce'], which, S['seed'] + k)
        base = test.loc[picks.uid]
        for X in S['inject_bytes']:
            dm = ant_floor.injection_delta(F['matcher'], F['sce'], picks, cls, X)
            for c in METHODS:
                d = dm if c == 'u_matcher' else (np.zeros(len(picks)) if attack == 'model_call' else np.full(len(picks), X))
                rows.append(dict(common, method=c, threshold=lam[c], attack=attack, bytes=float(X), runs=len(picks),
                                 caught=int((base[c].to_numpy() + d > lam[c]).sum())))
INJ = pd.DataFrame(rows)
TH = INJ[INJ.attack == 'none'].pivot_table(index='fold', columns='method', values='threshold')
print('thresholds (unexplained upload bytes per run) at a', S['alpha'], 'false-interruption level:')
display(TH.join(INJ[INJ.attack == 'none'].groupby('fold').cal_runs.first()).round(0))
print('folds with no finite threshold (too few calibration runs):', int(np.isinf(TH.to_numpy()).any(axis=1).sum()))
INJ.to_csv('reports/ant/injection_01.csv', index=False)
INJ_P = INJ.groupby(['attack', 'bytes', 'method'])[['runs', 'caught']].sum()
INJ_P['caught_share'] = INJ_P.caught / INJ_P.runs
display(INJ_P.round(3).unstack('method'))

## Stage F. Marks, figure, decision and log

In [ ]:
import matplotlib.pyplot as plt
MK = pd.DataFrame([
    dict(mark='G0 (as fixed)', decides=False, passed=G0_PASS,
         value=f"extracted {G0V['extracted_share']:.4f}, aligned {G0V['aligned_episode_share']:.4f}, "
               f"named {G0V['named_tcp443_share']:.4f}, listed {len(EP)} ({int(BOTH.sum())} with both files)"),
    dict(mark='G0r (revision 01)', decides=True, passed=G0R_PASS,
         value=f"extracted {G0V['extracted_share']:.4f}, aligned (real record times) "
               f"{G0V['aligned_episode_share_real_times']:.4f}, named {G0V['named_tcp443_share']:.4f}"),
    dict(mark='G1', decides=True, passed=G1_PASS, value='; '.join(f'{a}: {BACKEND.get(a) or "none"}' for a in N_EP.index)),
    dict(mark='M1', decides=True, passed=bool(M1[2] < 0), value='%.4f [%.4f, %.4f]' % M1 + f' over {len(FOLDS)} folds'),
    dict(mark='M2', decides=True, passed=bool(M2[0] <= S['m2_max']), value='%.4f [%.4f, %.4f]' % M2),
    dict(mark='M3', decides=True, passed=bool(M3[1] > 0), value='%.4f [%.4f, %.4f]' % M3 + f' over {len(FOLDS)} folds'),
])[['mark', 'value', 'passed', 'decides']]
MK.to_csv('reports/ant/marks_01.csv', index=False)
DECISION = 'PASS' if MK[MK.decides].passed.all() else 'FAIL'
failed = ', '.join(MK.mark[MK.decides & ~MK.passed])

PAL = {'u_matcher': '#2a78d6', 'u_task': '#eb6834', 'u_global': '#1baf7a'}
LABEL = {'u_matcher': 'step-level matcher', 'u_task': 'per-task allowlist', 'u_global': 'global allowlist'}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
lg = FLOOR[FLOOR.part == 'logged_step'].set_index('agent')
order = [a for a in AGENTS if a in lg.index] + ['claude_code', 'pooled']
y = np.arange(len(order))
for i, c in enumerate(METHODS):
    ax1.barh(y + (i - 1) * 0.26, lg.loc[order, c + '_share'], height=0.24, color=PAL[c], label=LABEL[c])
ax1.axvline(S['m2_max'], color='#52514e', lw=1, ls=':')
ax1.set_yticks(y, order); ax1.invert_yaxis()
ax1.set_xlabel('unexplained share of benign upload bytes inside logged steps')
ax1.set_title('Benign floor, held-out agent folder', loc='left')
ax1.legend(frameon=False, loc='lower right'); ax1.grid(axis='x', color='#e6e5e0', lw=0.8); ax1.set_axisbelow(True)
from matplotlib.lines import Line2D
STYLE = {'model_call': ('-', 'o'), 'new_host': ('--', 's')}
for c in METHODS:
    for attack, (ls, mk) in STYLE.items():
        d = INJ_P.loc[attack].xs(c, level='method')
        ax2.plot(d.index, d.caught_share, ls=ls, marker=mk, ms=6, lw=2, color=PAL[c])
ax2.axhline(S['alpha'], color='#52514e', lw=1, ls=':')
handles = ([Line2D([], [], color=PAL[c], lw=2, label=LABEL[c]) for c in METHODS]
           + [Line2D([], [], color='#52514e', lw=2, ls=ls, marker=mk, ms=6, label=a.replace('_', ' '))
              for a, (ls, mk) in STYLE.items()])
ax2.set_xscale('log'); ax2.set_ylim(-0.02, 1.02)
ax2.set_xlabel('injected upload bytes per run'); ax2.set_ylabel('share of test runs caught')
ax2.set_title('Simulated injections at a 5% false-interruption threshold', loc='left')
ax2.legend(handles=handles, frameon=False, fontsize=8, loc='upper left'); ax2.grid(color='#e6e5e0', lw=0.8); ax2.set_axisbelow(True)
for ax in (ax1, ax2):
    for s_ in ('top', 'right'):
        ax.spines[s_].set_visible(False)
plt.tight_layout(); plt.savefig('figures/ant/floor_01.png', dpi=160); plt.show()

TEXT = (f'DECISION: {DECISION}' + (f' (failed: {failed})' if failed else '') + '\n\n' + MK.to_string(index=False)
        + '\n\nRevision 01 (reports/ant/revision_01.json): ' + REVISION['decision']
        + '\n\nUsable runs per folder:\n' + USAGE.round(3).to_string()
        + '\n\nModel endpoints:\n' + BK.to_string(index=False)
        + '\n\nUnexplained share of upload bytes, logged steps and all parts:\n' + show.round(4).to_string()
        + "\n\nANT's own split against instance-disjoint folds:\n"
        + ANTP[ANTP.part.isin(['logged_step', 'all'])].round(4).to_string(index=False)
        + '\n\nSimulated injections (pooled):\n' + INJ_P.round(3).unstack('method').to_string())
with open('reports/ant/decision_01.txt', 'w') as f:
    f.write(TEXT + '\n')
reslog.log_result('01 ANT benign floor and go/no-go gate', TEXT)
print(TEXT)

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", "nb01: ANT benign floor and go/no-go gate: captures reduced to packet and flow tables (data/ant, not committed), model endpoint per agent folder, step-level matcher against per-task and global allowlists in 10 instance-disjoint leave-one-agent-out folds and ANT's own split, step-type shuffle check, simulated injections; marks fixed in Stage A; revision_01 after the probe: qwen records unusable for timing (479 runs without records, 116 with placeholder times), qwen left out of the step-level stages, G0 reported and replaced by G0r in the decision; reports/ant/*_01.csv, decision_01.txt, figures/ant/floor_01.png"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)